In [7]:
import torch
from datasets import Dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
source_texts = []
target_texts = []

with open("/content/sample_data/para-nmt-50m-small.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            source_texts.append("paraphrase: " + parts[0])
            target_texts.append(parts[1])

data_dict = {"source": source_texts, "target": target_texts}
dataset = Dataset.from_dict(data_dict).train_test_split(test_size=0.1)

In [9]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
def preprocess_function(examples):
    model_inputs = tokenizer(examples["source"], max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(text_target=examples["target"], max_length=128, truncation=True, padding="max_length")

    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/90000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [11]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_dir="./logs",
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [12]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
)

In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,1.338400,1.222848
2,1.211690,1.191045
3,1.130442,1.183955


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=33750, training_loss=1.2563729456018518, metrics={'train_runtime': 6298.3213, 'train_samples_per_second': 42.869, 'train_steps_per_second': 5.359, 'total_flos': 9135571599360000.0, 'train_loss': 1.2563729456018518, 'epoch': 3.0})

In [14]:
model.save_pretrained("./my_paraphrase_model")
tokenizer.save_pretrained("./my_paraphrase_model")
print("Model training complete and saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model training complete and saved!


In [15]:
model_path = "./my_paraphrase_model"
tokenizer = T5Tokenizer.from_pretrained(model_path)
model = T5ForConditionalGeneration.from_pretrained(model_path)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [25]:
def paraphrase(text):
    input_text = "paraphrase: " + text
    inputs = tokenizer(input_text, return_tensors="pt", max_length=128, truncation=True)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=128,
        num_beams=5,
        num_return_sequences=3,
        no_repeat_ngram_size=2,
        temperature=0.7,
        do_sample=True,
        early_stopping=True
    )

    results = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]
    return results


In [27]:
test_sentence = "Two small patches of swarmp forests still exist in the area of which one is in Chatla beel and the other near the village of Kalikrishnapur."
variations = paraphrase(test_sentence)

print(f"\nOriginal: {test_sentence}\n")
print("Generated Structural Alternatives:")
for i, variant in enumerate(variations, 1):
    print(f" {i}. {variant}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]


Original: Two small patches of swarmp forests still exist in the area of which one is in Chatla beel and the other near the village of Kalikrishnapur.

Generated Structural Alternatives:
 1. there are still two small swarmp forests in the area where one is in chatla beel and the other is near the village of kalikrishnapur.
 2. there are two small areas of swarmp forests in the area where one is in chatla beel and the other near the village of Kalikrishnapur.
 3. there are two small areas of swarmp forests in the area where one is in chatla beel and the other is near the village of Kalikrishnapur.


In [18]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [19]:
def get_sentence_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", max_length=128, truncation=True, padding=True).to(device)

    with torch.no_grad():
        encoder_outputs = model.encoder(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        hidden_states = encoder_outputs.last_hidden_state
        input_mask_expanded = inputs["attention_mask"].unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)

        sentence_vector = (sum_embeddings / sum_mask).squeeze().cpu().numpy()

    return sentence_vector

In [20]:
human_scores = []
predicted_similarities = []

test_file_path = "/content/sample_data/annotated-ppdb-test"

print(f"\nReading evaluation dataset from: {test_file_path}...")
with open(test_file_path, "r", encoding="utf-8") as f:
    for line_idx, line in enumerate(f, 1):
        parts = line.strip().split("\t")

        if len(parts) >= 3:
            try:
                score = float(parts[0])
                sent1 = parts[1]
                sent2 = parts[2]
            except ValueError:
                score = None
                text_segments = []

                for part in parts:
                    try:
                        val = float(part)
                        if score is None:
                            score = val
                    except ValueError:
                        if part.strip():
                            text_segments.append(part)

                if score is not None and len(text_segments) >= 2:
                    sent1 = text_segments[0]
                    sent2 = text_segments[1]
                else:
                    continue

            vec1 = get_sentence_embedding(sent1)
            vec2 = get_sentence_embedding(sent2)

            sim = cosine_similarity(vec1.reshape(1, -1), vec2.reshape(1, -1))[0][0]

            human_scores.append(score)
            predicted_similarities.append(sim)

spearman_corr, _ = spearmanr(human_scores, predicted_similarities)


Reading evaluation dataset from: /content/sample_data/annotated-ppdb-test...


In [21]:
print("\n================ EVALUATION RESULTS ================")
print(f"Total Sentence Pairs Processed: {len(human_scores)}")
print(f"Spearman Correlation ($\rho$): {spearman_corr:.4f}")
print("====================================================")


================ EVALUATION RESULTS ================
Total Sentence Pairs Processed: 1000
ho$): 0.6023
